### OBAC reasoning evaluation 

P1- Read Action, View scope, Permission grant for tables; Order (t001),  Transaction(t002), Account(t003), Loan(t004), Client(t006), Card(t007)

P2- Read Action, process scope, prohibition grant for columns; account_id(c002), account_to(c004), k_symbol(c006), order_id(c001), trans_id(c007), k_symbol(c032), loan_id(c016), client_id(c021)

P3- Read action, process scope, permission grant for columns; amount(c005), date(c008), type(c009), district_id(c014), data(c033), amount(c031), date(c034), amount(c035)

P4- Read action, view scope, permission grant for columns; all the other column in permitted tables

P5- Read action, process scope, prohibition grant for table; District(t008), Disposition(t008) 

P6- Modify action, Insert scope, permission grant for table; District(t008)

P7- Modify action, Update scope, permission grant for table; Card(t007)

P8- Modify action, Delete scope, permission grant for table; Client(t006)

| Policy ID | Action type | Action scope | Permission type | Permission level | Object | 
| ---       | ---         | ---          | ---             | ---              | ---    |
| p1        | Read        | View         | Permission      | Table            | Order (t001),  Transaction(t002), Account(t003), Loan(t004) | 
| P2        | Read        | Process      | Prohibition     | Column           | account_id(c002), account_to(c004), k_symbol(c006) |
| p3        | Read        | Process      | Permission      | Column           | amount(c005), date(c008), type(c009) |
| p4        | Read        | View         | Permission      | Column           | bank_to(c003), operation(c010), balance(c011), bank(c012)  |
| p5        | Read        | Process      | Prohibition     | Table            | Client(t006), Card(t007), Disposition(t008) |
| p6        | Read        | Process      | Permission      | Row              | district_id (c014) == "main" |
| p7        | Modify      | Insert       | Permission      | Table            | District(t008) |
| p8        | Modify      | Update       | Permission      | Table            | Card(t007) |
| p9        | Modify      | Delete       | Permission      | Table            | Client(t006) |

_TODO: give read access to Client table > this is cover a scenario where a table have more than one valid access type_ \
__[Design note:] I think we need to define table of the column in the Row level access control. For that we need to have a rule format. But I don't think this would matter in the UI based access control definition.__ 

---
__Updates__  \
- We only works with the prohibition grants and conditional grants, even though user / admin could create permission grants. This because we cannot use permission grants in SWRL rule because of SWRL does not support negation. \
- Also, we accept `condition` variable at Row level policies and, 
- We added `comment` string field to track policies.

Here is the updated policy table 

| Policy_id | Agent ID | Action Type | Action Scope | Grant Type | Grant Level | Objects | Condition | Comment |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| p001 | a0012 | Read | Process | Prohibited | Table | Disposition (t005), District (t008) | - | table that agent couldn't read from |
| p002 | a0012 | Modify | Insert | Prohibited | Table | Order (t001), Loan(t004), Account(t003), Transaction (t002), Disposition (t005), Card (t007), Client (t006) | - | table that agent prohibited to do insert operation |
| p003 | a0012 | Modify | Update | Prohibited | Table | Order (t001), Loan(t004), Account(t003), Transaction (t002), Disposition (t005), District (t008), Client (t006) | - | table that agent prohibited to do update operation |
| p004 | a0012 | Modify | Delete | Prohibited | Table | Order (t001), Loan(t004), Account(t003), Transaction (t002), Disposition (t005), Card (t007), District (t008) | - | table that agent prohibited to do delete operation |
| p005 | a0012 | Read | Process | Prohibited | Column | order_id(c001), account_to(c004), k_symbol(c006), account_id(c002), k_symbol(c032), load_id(c016), client_id(c021) | - | columns that agent could not access in the tables that agent can access | 
| p006 | a0012 | Read | View | Prohibited | Column | amount (c005), district_id (c014), date (c033), date (c008), type (c009), amount (c031), date (c034), amount (c035) | - | columns that agent could only used to process operation in the read actions in the tables that agent can read from | 
| p007 | a0012 | Read | View | Conditional | Row | Client (t006) | district_id = "d0987" | rows that agent accepted to access from the Client table | 

## Valid Queries

In [ ]:
# simple SQL query to select bank and balance from Transaction table

query = "SELECT bank, balance FROM Transaction;"

In [ ]:
# updated SQL query to functional aggregation 'amount' column which has process access only

query = "SELECT SUM(amount) AS total_order_amount FROM Order;"

In [ ]:
# generate a valid little advance SQL query 

query = "SELECT loan_id, payment, status FROM Loan WHERE amount > 1000 AND status = 'active';"

## Invalid Queries 

In [ ]:
# SQL query accessing columns that violate process access, > account_id and account_to violate p2 | amount violates p3
# q002
query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

In [ ]:
# SQL query trying to view for process permission column violating p3
# q004
query = "SELECT DISTINCT(date) FROM Transaction; "

In [ ]:
# SQL query to view column which has no access by violating p3

query = "SELECT account_id, SUM(amount) AS daily_amount FROM Order GROUP BY account_id; "

In [ ]:
# SQL query accessing columns from prohibited table violating p5 

query = "SELECT client_id, gender FROM Client;"